# 03 ออกแบบ split และ preprocessing
สมาชิกเลือกและเขียนวิธีแบ่งข้อมูล/เตรียมข้อมูลเองหลังเข้าใจ dataset ยังไม่ล็อกสัดส่วน split หรือวิธีเติมค่าให้

จุดประสงค์: ประเมินให้ตรงกับการใช้งานและป้องกันข้อมูลรั่ว


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/Bicycle_Rental_Prediction')]
ROOT = next((p for p in candidates if (p / 'config/project.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('เปิด notebook จาก repository หรือทำขั้นตอนเตรียม Colab ใน START_HERE_TH ก่อน')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from starter_support import load_config, configuration_problems, resolve_dataset_path
config = load_config(ROOT)
problems = configuration_problems(config)
if problems:
    raise ValueError('\n'.join(problems))
print('อ่าน config ได้แล้ว ค่า null เป็นสถานะรอยืนยัน ไม่ใช่ข้อมูลที่ต้องเดา')


In [ ]:
df = None
path = resolve_dataset_path(config, ROOT)
if path is None:
    print('ยังไม่ได้ตั้ง dataset.path: อ่านโจทย์และคำถามก่อนได้ ยังไม่มีข้อมูลให้วิเคราะห์')
elif not path.is_file():
    print('ยังไม่พบไฟล์: วาง dataset จากอาจารย์ใน data/raw ตาม config')
else:
    import pandas as pd
    df = pd.read_csv(path, encoding=config['dataset']['encoding'],
                     sep=config['dataset']['delimiter'])
    print('อ่านไฟล์ได้แล้ว ต้องตรวจความหมายและคุณภาพก่อนใช้งาน')


## ตกลงคำถามก่อนเขียน split
- ประเมินตามเงื่อนไข หรือทำนายล่วงหน้าเท่าใด
- เวลาใดถือเป็นเวลาที่ออกคำทำนาย และ feature แต่ละตัวพร้อมหรือยัง
- มีหลายสถานีหรือหลายแถวต่อเวลาหรือไม่ ต้องแยก entity ตามคำถามใด
- จะกัน test และใช้ validation อย่างไร ต้องรักษาลำดับเวลาหรือไม่

เขียนช่วงวันที่ จำนวนแถว และกฎกันรายการซ้ำ ให้เพื่อนตรวจว่าข้อมูลอนาคตหรือแถวเดียวกันไม่หลุดข้ามชุด


In [ ]:
# เขียนวิธีแบ่ง train / validation / test ของกลุ่มเอง
# ให้เห็นช่วงเวลาและจำนวนแถวของแต่ละชุด เก็บ test ไว้สำหรับรอบสุดท้าย


## ออกแบบ preprocessing
เลือกวิธี missing/category/scaling ตามข้อมูลและอัลกอริทึม fit เฉพาะ train แล้ว transform validation/test ด้วยขั้นตอนเดิม ถ้าทำ cross-validation ต้อง fit ใหม่ใน train ของแต่ละ fold

พิจารณา ColumnTransformer และ Pipeline เพื่อรวมขั้นตอน แต่ต้องอธิบายแต่ละขั้นได้ ห้าม fit preprocessing บนทั้ง df ก่อนแบ่งชุด

เอกสาร: [common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html), [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)


In [ ]:
# เขียน preprocessing ของกลุ่มเอง
# เพิ่ม checks ที่แสดงว่า target/outcome components ไม่อยู่ใน X
# จัดการ feature order และหมวดหมู่ที่ไม่เคยเห็นอย่างชัดเจน


## สิ่งส่งต่อและจุดตรวจ
เพื่อนอีกคนตรวจแผนแบ่งข้อมูลและลำดับ fit/transform จากโค้ดจริง บันทึกเหตุผลและข้อจำกัดใน decisions ไม่ใช้คำว่าป้องกัน leakage แล้วหากยังไม่ตรวจโค้ด


## ก่อนส่งต่อ
เก็บข้อสังเกตและหลักฐานที่เผยแพร่ได้ไว้ใน evidence/results ให้เพื่อนอีกคนตรวจและลองทำซ้ำ จากนั้นล้าง outputs และ execution counts ก่อน commit notebook อย่าใส่ผลสมมติเป็นผลทดลองจริง
